In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
n = 250
area = np.random.randint(600, 3500, size=n)
bedrooms = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.1, 0.35, 0.35, 0.15, 0.05])
age = np.random.randint(1, 30, size=n)
noise = np.random.normal(0, 25000, size=n)
price = (50000 + area * 185 + bedrooms * 15000 - age * 1800 + noise).astype(int)

df = pd.DataFrame({'area_sqft': area, 'bedrooms': bedrooms, 'age_years': age, 'price_usd': price})
mask = np.random.rand(*df.shape) < 0.05
df_missing = df.mask(mask)
df_missing.to_csv('housing_data_raw.csv', index=False)

In [ ]:
df_raw = pd.read_csv('housing_data_raw.csv')
print("Shape:", df_raw.shape)
print("Column Types:\n", df_raw.dtypes)
print("Missing values count:\n", df_raw.isnull().sum())
df_clean = df_raw.copy()
for col in df_clean.columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

In [ ]:
stats = df_clean[['area_sqft', 'bedrooms', 'age_years', 'price_usd']].agg(['mean', 'median', 'std', 'min', 'max']).T
print("Summary Statistics:\n", stats)

corr = df_clean[['area_sqft', 'bedrooms', 'age_years', 'price_usd']].corr()
print("Correlation Matrix:\n", corr)

avg_price_bed = df_clean.groupby('bedrooms')['price_usd'].mean().reset_index()
print("Average Price per Bedroom Count:\n", avg_price_bed)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df_clean[['area_sqft', 'bedrooms', 'age_years']]
y = df_clean['price_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
sample_house = pd.DataFrame({'area_sqft': [1200], 'bedrooms': [3], 'age_years': [10]})
predicted_val = model.predict(sample_house)[0]

print(f"Predicted Price (1,200 sq ft, 3 bed, 10 yrs old): ${predicted_val:,.2f}")

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(x=y_test, y=y_pred, alpha=0.8, color='#2b5c8f', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Actual vs Predicted Prices')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
sns.histplot(df_clean['price_usd'], kde=True, color='#2b8f62', bins=20)
plt.xlabel('Price ($)')
plt.title('Price Distribution')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

X_train_b = np.c_[np.ones((len(X_train), 1)), X_train]
theta_best = np.linalg.inv(X_train_b.T.dot(X_train_b)).dot(X_train_b.T).dot(y_train)

print("\n--- Normal Equation vs Scikit-Learn Comparison ---")
print("Normal Equation Coefficients:", np.round(theta_best, 2))
print("Scikit-learn Intercept & Coefs:", np.round([model.intercept_] + list(model.coef_), 2))